# Солнцезащитная плёнка с молекулярным накоплением солнечной энергии (MOST + UV)
## Автономный запуск полного пайплайна в Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suharevalexey/Case/blob/main/MOST_UV_End_to_End_Colab.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-suharevalexey%2FCase-blue.svg)](https://github.com/suharevalexey/Case)

Данный блокнот полностью автономен и оптимизирован для выполнения в Google Colab в 1 клик.
Все необходимые файлы, веса 12 моделей и результаты автоматически подгружаются напрямую из GitHub.


In [ ]:
# 1. Установка химического стека и автоматическая загрузка проекта из GitHub
import os, sys, shutil

# Установка зависимостей в виртуальное окружение Colab
print("📦 Установка химического стека (RDKit, XGBoost)...")
!pip install -q rdkit xgboost scikit-learn matplotlib pandas openpyxl

# Определение пути к репозиторию (без смены рабочего каталога %cd!)
REPO_DIR = "/content/Case" if os.path.exists("/content") else os.path.abspath("./Case")

if not os.path.exists(os.path.join(REPO_DIR, "models", "evaluators")):
    print("🚀 Загрузка актуального репозитория из GitHub...")
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR, ignore_errors=True)
    !git clone --depth 1 https://github.com/suharevalexey/Case.git {REPO_DIR}
else:
    print("🔄 Обновление существующего репозитория...")
    !git -C {REPO_DIR} pull origin main

# Добавление исходного кода в системный путь Python
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Очистка кэша импортов Python для гарантированной свежести модулей
for mod in list(sys.modules.keys()):
    if any(k in mod for k in ["evaluators", "evaluator_model", "applicability_domain", "features", "strategy_b0"]):
        del sys.modules[mod]

print("✅ Все модули, модели и зависимости успешно подключены напрямую из GitHub!")


In [ ]:
# 2. Загрузка и анализ результатов генеративной стратегии B0 (3 теста и сводка)
import os, pandas as pd

REPO_DIR = "/content/Case" if os.path.exists("/content") else os.path.abspath(".")
local_csv = os.path.join(REPO_DIR, "results", "metrics_summary_B0.csv")
github_url = "https://raw.githubusercontent.com/suharevalexey/Case/main/results/metrics_summary_B0.csv"
csv_path = local_csv if (os.path.exists(local_csv) and os.path.getsize(local_csv) > 0) else github_url

df_res = pd.read_csv(csv_path)
print("=" * 85)
print("  РЕЗУЛЬТАТЫ 3 НЕЗАВИСИМЫХ ТЕСТОВЫХ ЗАПУСКОВ B0 (Random Seeds: 42, 101, 2024)")
print("=" * 85)
cols_individual = [
    "method", "seed", "N_valid", "Validity", "Uniqueness", "Novelty", 
    "Internal_Diversity", "JSR_Oracle (Joint Success Rate)", 
    "Overall_Pass_Constraints", "SAScore_Mean", "AD_Mean_Dist_to_D_A", "AD_Mean_Dist_to_D_B"
]
df_tests = df_res.iloc[0:3][cols_individual]
print(df_tests.to_string(index=False))

print("\n" + "=" * 85)
print("  АГРЕГИРОВАННЫЕ СВОДНЫЕ МЕТРИКИ (Mean ± Std по 3 запускам, Раздел 6 Задания)")
print("=" * 85)
row_agg = df_res.iloc[3]

# Безопасное извлечение метрик с защитой от различий в наименованиях колонок
def find_stat(row, base_name, stat_type):
    target_suffix = f"_{stat_type}"
    for col in row.index:
        if col.startswith(base_name) and col.endswith(target_suffix):
            val = row[col]
            if pd.notna(val):
                return float(val)
    return 0.0

metrics_list = [
    ("Validity (Валидность)", "Validity"),
    ("Uniqueness (Уникальность)", "Uniqueness"),
    ("Novelty (Новизна)", "Novelty"),
    ("Internal Diversity (Разнообразие)", "Internal_Diversity"),
    ("JSR Oracle (Joint Success Rate)", "JSR_Oracle"),
    ("Overall Pass Constraints", "Overall_Pass_Constraints"),
    ("SAScore (Синтетическая сложность)", "SAScore_Mean"),
    ("AD Dist to D_A (MOST)", "AD_Mean_Dist_to_D_A"),
    ("AD Dist to D_B (UV/Skin)", "AD_Mean_Dist_to_D_B")
]

summary_data = []
for label, key in metrics_list:
    m_val = find_stat(row_agg, key, "mean")
    s_val = find_stat(row_agg, key, "std")
    summary_data.append({"Метрика": label, "Mean (Среднее)": f"{m_val:.4f}", "Std (Станд. откл.)": f"±{s_val:.4f}"})

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))
display(df_tests)


In [ ]:
# 3. Интерактивная оценка молекулы через PropertyEvaluatorSuite
import os, sys
from evaluators import PropertyEvaluatorSuite
from rdkit import Chem
from rdkit.Chem import Draw

REPO_DIR = "/content/Case" if os.path.exists("/content") else os.path.abspath(".")
suite = PropertyEvaluatorSuite(base_dir=REPO_DIR)

# Тестовая молекула: Авобензон (эталонный UV-фильтр)
test_smiles = "COc1ccc(C(=O)CC(=O)c2ccc(C(C)(C)C)cc2)cc1"
res = suite.evaluate_molecule(test_smiles)

print("=" * 85)
print(f"  РЕЗУЛЬТАТ ОЦЕНКИ МОЛЕКУЛЫ (7 ЦЕЛЕВЫХ СВОЙСТВ): {test_smiles}")
print("=" * 85)
print(f"  • Синтетическая доступность (SAScore):  {res['synthetic_accessibility']:.2f} (порог <= 4.5)")
print(f"  • Расстояние до домена A (MOST):        {res['dist_to_D_A']:.4f} (AD threshold <= 0.65)")
print(f"  • Расстояние до домена B (UV/Skin):     {res['dist_to_D_B']:.4f} (AD threshold <= 0.65)")
print(f"  • Группа A - Lambda Max (поглощение):   {res['pred_group_A_absorption_max_nm']:.1f} нм (unc: ±{res['unc_group_A_absorption_max_nm']:.1f}, цель: 290-400 нм)")
print(f"  • Группа A - Log Extinction (эпсилон):  {res['pred_group_A_log_extinction']:.2f} (unc: ±{res['unc_group_A_log_extinction']:.2f}, цель: >= 3.5)")
print(f"  • Группа A - Quantum Yield (Ф):         {res['pred_group_A_photochem_efficiency']:.3f} (unc: ±{res['unc_group_A_photochem_efficiency']:.3f})")
print(f"  • Группа A - Log t1/2 (время полураспада): {res['pred_group_A_log_half_life']:.2f} log10(c) (unc: ±{res['unc_group_A_log_half_life']:.2f}, цель: >= 3.56 [>=1ч])")
print(f"  • Группа B - Log Kp (проницаемость):    {res['pred_group_B_log_kp']:.2f} см/с (unc: ±{res['unc_group_B_log_kp']:.2f}, цель: <= -6.0)")
print(f"  • Группа B - Сенсибилизация кожи (LLNA): {res['pred_group_B_skin_sensitization']:.4f} (цель: <= 0.50)")
print(f"  • Группа B - Раздражение кожи:          {res['pred_group_B_skin_irritation']:.4f} (цель: <= 0.50)")
print(f"  • Прохождение суррогатных фильтров:     {'ДА' if res['pass_surrogate_all'] else 'НЕТ'}")
print(f"  • Прохождение независимых оракулов:     {'ДА' if res['pass_oracle_all'] else 'НЕТ'}")
print(f"  • Итоговый статус (Overall Pass):       {'УСПЕХ' if res['pass_constraints'] else 'ОТСЕВ'}")
print("=" * 85)

mol = Chem.MolFromSmiles(test_smiles)
img = Draw.MolToImage(mol, size=(450, 250))
display(img)


In [ ]:
# 4. Визуализация топ-кандидатов из сгенерированной выборки B0 (N=3000)
import os, pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw

REPO_DIR = "/content/Case" if os.path.exists("/content") else os.path.abspath(".")
local_gen = os.path.join(REPO_DIR, "results", "generated_B0.csv")
github_gen = "https://raw.githubusercontent.com/suharevalexey/Case/main/results/generated_B0.csv"
gen_path = local_gen if (os.path.exists(local_gen) and os.path.getsize(local_gen) > 0) else github_gen

df_gen = pd.read_csv(gen_path)
print(f"Всего сгенерировано молекул в выборке B0: {len(df_gen):,}")

# Фильтруем успешные кандидаты (прошедшие все ограничения Оракула и SAScore <= 5.0)
df_pass = df_gen[df_gen["pass_constraints"] == 1]
print(f"Кандидатов, прошедших все фильтры и Оракул: {len(df_pass):,} ({len(df_pass)/len(df_gen)*100:.2f}%)")

top_mols = [Chem.MolFromSmiles(s) for s in df_pass["SMILES"].head(8)]
legends = [f"#{i+1} SA={sa:.2f}" for i, sa in enumerate(df_pass["synthetic_accessibility"].head(8))]
img_grid = Draw.MolsToGridImage(top_mols, molsPerRow=4, subImgSize=(300, 200), legends=legends)
display(img_grid)


In [ ]:
# 5. Интерактивная генерация новой партии молекул-кандидатов (Стратегия B0)
from strategy_b0 import BaselineGeneratorB0

REPO_DIR = "/content/Case" if os.path.exists("/content") else os.path.abspath(".")
data_dir = os.path.join(REPO_DIR, "data", "processed")

print("🔬 Запуск стохастического кроссовера экзоциклических связей фрагментов D_A и D_B...")
generator = BaselineGeneratorB0(data_dir=data_dir)
new_smiles, attempts, val, uniq = generator.run_generation(seed=42, target_unique_count=50)

print("🧪 Оценка сгенерированной выборки через PropertyEvaluatorSuite (7 целевых свойств)...")
df_eval = suite.evaluate_batch(new_smiles)
pass_n = (df_eval["pass_constraints"] == 1).sum()
print(f"Успешно прошли все критерии Оракула и синтетической доступности: {pass_n} из {len(df_eval)} ({pass_n/len(df_eval)*100:.1f}%)\n")

display_cols = ["SMILES", "synthetic_accessibility", "dist_to_D_A", "dist_to_D_B", "pred_group_A_absorption_max_nm", "pred_group_A_log_half_life", "pred_group_B_log_kp", "pass_constraints"]
display(df_eval[display_cols].head(10))
